# 01_bronze_notebook_crb_billings

**Purpose**: Ingest all CRB billing raw Excel files from Lakehouse Files into Bronze Delta tables.

**Source**: `Files/billing/{source}/` (populated by `01_bronze_ingest_crb_billings` pipeline)

**Output tables** (all in `APAC_CRM_Analytics_bronze_LH`):

| Cell | Table | File | Sheet |
| :--- | :---- | :--- | :---- |
| 1 | `src_saiba_crb` | all xlsx in `saiba/` | `Sheet1` |
| 2 | `src_arias_crb` | `Arias Revenue.xlsx` | `Revenuev2` (header row 2) |
| 3 | `src_eclipse_crb` | `Eclipse Recurring Report.xlsm` | `RAW Data - 2018 -2019 Postings` (header row 2) |
| 4 | `src_eclipse_london` | 2 xlsx files in `eclipse_london/` | `Sheet1` |
| 5 | `src_eglobal_income_report` | `IncomebyDepartmentandClient*.xlsx` | `Source Data` (header row 3) |
| 6 | `src_eglobal_premium_report` | `Premium_Volume_Report*.xlsx` | `Detail Section` |
| 7 | `src_gswin_crb` | `GSWin Data.xlsx` | `PolicyReport` |
| 8 | `src_ret_oracle` | all xlsx in `oracle/` | `Outbound Retirement` |
| 9 | `src_wr_spm` | all xlsx in `spm/` | `Sheet1` |

**Saiba GCID logic (applied at Bronze):**
- Entry Date: GCID column already present in source
- RI Business: GCID = `RI_` + resolved CustName + `_` + S.NO_; CustName falls back to `Name of the Reinsured (Cedant)` when null
- Fee Business: GCID = CNo (customer number, Column B)

**Lakehouse**: Attach `APAC_CRM_Analytics_bronze_LH` as default before running.

In [ ]:
# =============================================================================
# Setup & Shared Config
# =============================================================================
import pandas as pd
import os

LAKEHOUSE  = "APAC_CRM_Analytics_bronze_LH"
FILES_BASE = "/lakehouse/default/Files/billing/"


def normalize_cols(df):
    # Match Dataflow Gen2: only a trailing period is replaced with underscore.
    # Commas are stripped -- Hive metastore rejects them even with column mapping.
    # NaN headers (blank Excel columns) are cast to str then dropped.
    def fix(col):
        s = str(col).replace(',', '')
        return s[:-1] + '_' if s.endswith('.') else s
    df.columns = [fix(col) for col in df.columns]
    # Drop unnamed columns — empty Excel headers that pandas names Unnamed: N
    df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]
    return df


def detect_header_row(fpath, sheet_name, max_search=10):
    # Peek first max_search rows without headers and find the row with the
    # most text-like (non-numeric, non-empty) values — that is the header row.
    # Handles files where row 0 is a numeric aggregate rather than column names.
    df_peek = pd.read_excel(fpath, sheet_name=sheet_name, header=None, dtype=str, nrows=max_search)
    best_row, best_score = 0, 0
    for i, row in df_peek.iterrows():
        score = sum(
            1 for v in row.tolist()
            if str(v) not in ("nan", "None", "")
            and not str(v).replace(".", "").replace("-", "").replace(" ", "").isdigit()
        )
        if score > best_score:
            best_score = score
            best_row = i
    return best_row


def inspect_sheets(folder, ext=".xlsx"):
    """Print sheet names for every file in folder — use to diagnose wrong sheet."""
    for fname in sorted(os.listdir(folder)):
        if fname.endswith(ext) and not fname.startswith("~$"):
            xl = pd.ExcelFile(os.path.join(folder, fname))
            print(f"{fname}: {xl.sheet_names}")


def write_table(df_spark, table_name):
    full_name = f"{LAKEHOUSE}.{table_name}"
    # Column mapping preserves spaces and special characters in column names,
    # matching the behaviour of Dataflow Gen2 which enables it by default.
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .saveAsTable(full_name)
    )
    count = spark.sql(f"SELECT COUNT(*) FROM {full_name}").collect()[0][0]
    print(f"Written: {count} rows -> {full_name}")


print("Setup complete.")

In [ ]:
# =============================================================================
# Cell 1: Saiba
# Files  : all xlsx in billing/saiba/
# Sheet  : Sheet1
# Header : auto-detected per file (Fee Business has metadata rows before headers)
# Output : src_saiba_crb
#
# GCID enrichment applied per file:
#   Entry Date   -> GCID column already present, no change
#   RI Business  -> GCID = "RI_" + CustName + "_" + S.NO_
#                   CustName falls back to Name of the Reinsured (Cedant) when null
#   Fee Business -> GCID = CNo
# =============================================================================
CEDANT_COL = "Name of the Reinsured (Cedant)"

folder = f"{FILES_BASE}saiba/"
dfs = []
for fname in sorted(os.listdir(folder)):
    if fname.endswith(".xlsx") and not fname.startswith("~$"):
        fpath = os.path.join(folder, fname)
        header_row = detect_header_row(fpath, "Sheet1")
        df_tmp = pd.read_excel(fpath, sheet_name="Sheet1", header=header_row, dtype=str)
        df_tmp["Source_Name"] = fname
        df_tmp = normalize_cols(df_tmp)

        if "RI Business" in fname:
            cust = df_tmp["CustName"].fillna("").astype(str).str.strip()
            cedant = (
                df_tmp[CEDANT_COL].fillna("").astype(str).str.strip()
                if CEDANT_COL in df_tmp.columns
                else pd.Series("", index=df_tmp.index)
            )
            resolved = cust.where(~cust.isin(["", "nan"]), cedant)
            df_tmp["CustName"] = resolved
            s_no = df_tmp["S.NO_"].fillna("").astype(str).str.strip()
            df_tmp["GCID"] = "RI_" + resolved + "_" + s_no

        elif "Fee Business" in fname:
            df_tmp["GCID"] = df_tmp["CNo"].fillna("").astype(str).str.strip()

        print(f"Loaded {fname}: {len(df_tmp)} rows, {len(df_tmp.columns)} cols (header at row {header_row})")
        dfs.append(df_tmp)

if not dfs:
    raise ValueError(f"No xlsx files found in {folder}")

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_saiba = spark.createDataFrame(df_pandas)
df_saiba.printSchema()
display(df_saiba.limit(3))

write_table(df_saiba, "src_saiba_crb")

In [ ]:
# =============================================================================
# Cell 2: Arias
# File   : billing/arias/Arias Revenue.xlsx
# Sheet  : Revenuev2  |  Header: row 2 (1 title row skipped)
# Output : src_arias_crb
# =============================================================================
file_path = f"{FILES_BASE}arias/Arias Revenue.xlsx"

df_pandas = pd.read_excel(file_path, sheet_name="Revenuev2", header=1, dtype=str)
df_pandas = normalize_cols(df_pandas)

print(f"Rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_arias = spark.createDataFrame(df_pandas)
df_arias.printSchema()
display(df_arias.limit(3))

write_table(df_arias, "src_arias_crb")

In [ ]:
# =============================================================================
# Cell 3: Eclipse CRB
# File   : billing/eclipse_crb/Eclipse Recurring Report.xlsm
# Sheet  : RAW Data - 2018 -2019 Postings  |  Header: row 2 (1 title row skipped)
# Output : src_eclipse_crb
# =============================================================================
file_path = f"{FILES_BASE}eclipse_crb/Eclipse Recurring Report.xlsm"

df_pandas = pd.read_excel(
    file_path,
    sheet_name="RAW Data - 2018 -2019 Postings",
    header=1,
    dtype=str,
    engine="openpyxl"
)
df_pandas = normalize_cols(df_pandas)

print(f"Rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_eclipse_crb = spark.createDataFrame(df_pandas)
df_eclipse_crb.printSchema()
display(df_eclipse_crb.limit(3))

write_table(df_eclipse_crb, "src_eclipse_crb")

In [ ]:
# =============================================================================
# Cell 4: Eclipse London
# Files  : 2 specific xlsx in billing/eclipse_london/
# Sheet  : Sheet1  |  Header: row 1
# Output : src_eclipse_london
# =============================================================================
eclipse_london_files = [
    "Eclipse_Asia Client and Asia Insurer Reporting _MIR_12047_PowerBI File.xlsx",
    "Eclipse_Non Asia Client and Asia Insurer Reporting _MIR_12046_PowerBI File.xlsx"
]

folder = f"{FILES_BASE}eclipse_london/"
dfs = []
for fname in eclipse_london_files:
    fpath = os.path.join(folder, fname)
    df_tmp = pd.read_excel(fpath, sheet_name="Sheet1", header=0, dtype=str)
    df_tmp["Source_Name"] = fname
    df_tmp = normalize_cols(df_tmp)
    print(f"Loaded {fname}: {len(df_tmp)} rows, {len(df_tmp.columns)} cols")
    dfs.append(df_tmp)

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_eclipse_london = spark.createDataFrame(df_pandas)
df_eclipse_london.printSchema()
display(df_eclipse_london.limit(3))

write_table(df_eclipse_london, "src_eclipse_london")

In [ ]:
# =============================================================================
# Cell 5: eGlobal Income
# Files  : billing/eglobal_income/IncomebyDepartmentandClient*.xlsx
# Sheet  : Source Data  |  Header: row 3 (2 title rows skipped)
# Output : src_eglobal_income_report
# =============================================================================
folder = f"{FILES_BASE}eglobal_income/"
dfs = []
for fname in sorted(os.listdir(folder)):
    if (
        fname.startswith("IncomebyDepartmentandClient")
        and fname.endswith(".xlsx")
        and not fname.startswith("~$")
    ):
        fpath = os.path.join(folder, fname)
        df_tmp = pd.read_excel(fpath, sheet_name="Source Data", header=2, dtype=str)
        df_tmp["Source_Name"] = fname
        print(f"Loaded {fname}: {len(df_tmp)} rows")
        dfs.append(df_tmp)

if not dfs:
    raise ValueError(f"No IncomebyDepartmentandClient files found in {folder}")

df_pandas = pd.concat(dfs, ignore_index=True)
df_pandas = normalize_cols(df_pandas)

print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_eglobal_income = spark.createDataFrame(df_pandas)
df_eglobal_income.printSchema()
display(df_eglobal_income.limit(3))

write_table(df_eglobal_income, "src_eglobal_income_report")

In [ ]:
# =============================================================================
# Cell 6: eGlobal Premium
# Files  : billing/eglobal_premium/Premium_Volume_Report*.xlsx
# Sheet  : Detail Section  |  Header: row 1
# Output : src_eglobal_premium_report
# =============================================================================
folder = f"{FILES_BASE}eglobal_premium/"
dfs = []
for fname in sorted(os.listdir(folder)):
    if (
        fname.startswith("Premium_Volume_Report")
        and fname.endswith(".xlsx")
        and not fname.startswith("~$")
    ):
        fpath = os.path.join(folder, fname)
        df_tmp = pd.read_excel(fpath, sheet_name="Detail Section", header=0, dtype=str)
        df_tmp["Source_Name"] = fname
        df_tmp = normalize_cols(df_tmp)
        print(f"Loaded {fname}: {len(df_tmp)} rows, {len(df_tmp.columns)} cols")
        dfs.append(df_tmp)

if not dfs:
    raise ValueError(f"No Premium_Volume_Report files found in {folder}")

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_eglobal_premium = spark.createDataFrame(df_pandas)
df_eglobal_premium.printSchema()
display(df_eglobal_premium.limit(3))

write_table(df_eglobal_premium, "src_eglobal_premium_report")

In [ ]:
# =============================================================================
# Cell 7: GSWin
# File   : billing/gswin/GSWin Data.xlsx
# Sheet  : PolicyReport  |  Header: row 1
# Output : src_gswin_crb
# =============================================================================
file_path = f"{FILES_BASE}gswin/GSWin Data.xlsx"

df_pandas = pd.read_excel(file_path, sheet_name="PolicyReport", header=0, dtype=str)
df_pandas = normalize_cols(df_pandas)

print(f"Rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_gswin = spark.createDataFrame(df_pandas)
df_gswin.printSchema()
display(df_gswin.limit(3))

write_table(df_gswin, "src_gswin_crb")

In [ ]:
# =============================================================================
# Cell 8: Oracle RET
# Files  : all xlsx in billing/oracle/
# Sheet  : Outbound Retirement  |  Header: row 1
# Output : src_ret_oracle
# =============================================================================
folder = f"{FILES_BASE}oracle/"
dfs = []
for fname in sorted(os.listdir(folder)):
    if fname.endswith(".xlsx") and not fname.startswith("~$"):
        fpath = os.path.join(folder, fname)
        df_tmp = pd.read_excel(fpath, sheet_name="Outbound Retirement", header=0, dtype=str)
        df_tmp["Source_Name"] = fname
        print(f"Loaded {fname}: {len(df_tmp)} rows")
        dfs.append(df_tmp)

if not dfs:
    raise ValueError(f"No xlsx files found in {folder}")

df_pandas = pd.concat(dfs, ignore_index=True)
df_pandas = normalize_cols(df_pandas)
# Strip any trailing underscores introduced by normalize_cols on names ending with punctuation
df_pandas.columns = [col.rstrip('_') for col in df_pandas.columns]

print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_ret_oracle = spark.createDataFrame(df_pandas)
df_ret_oracle.printSchema()
display(df_ret_oracle.limit(3))

write_table(df_ret_oracle, "src_ret_oracle")

In [ ]:
# =============================================================================
# Cell 9: SPM (WR)
# Files  : all xlsx in billing/spm/ (excludes ~$ temp files)
# Sheet  : Sheet1  |  Header: row 1
# Output : src_wr_spm
# =============================================================================
folder = f"{FILES_BASE}spm/"
dfs = []
for fname in sorted(os.listdir(folder)):
    if fname.endswith(".xlsx") and not fname.startswith("~$"):
        fpath = os.path.join(folder, fname)
        try:
            df_tmp = pd.read_excel(fpath, sheet_name="Sheet1", header=0, dtype=str)
            df_tmp["Source_Name"] = fname
            df_tmp = normalize_cols(df_tmp)
            print(f"Loaded {fname}: {len(df_tmp)} rows, {len(df_tmp.columns)} cols")
            dfs.append(df_tmp)
        except Exception as e:
            print(f"Warning: could not read {fname}: {e}")

if not dfs:
    raise ValueError(f"No xlsx files found in {folder}")

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_pandas)} | Cols: {len(df_pandas.columns)}")
print(list(df_pandas.columns))

df_wr_spm = spark.createDataFrame(df_pandas)
df_wr_spm.printSchema()
display(df_wr_spm.limit(3))

write_table(df_wr_spm, "src_wr_spm")